<a href="https://colab.research.google.com/github/AjayCunanan/prompt-engineering-practice/blob/main/01_prompt_chaining_customer_support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install -q google-genai

import time
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-2.5-flash"

def ask_llm(prompt, temperature=0.3):
    time.sleep(13)  # free tier allows 5 requests/min, so wait between calls
    config = types.GenerateContentConfig(temperature=temperature)
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    return response.text

print("Ready!")

Ready!


In [2]:
reply = ask_llm("In one sentence, what is a customer support chatbot?")
print(reply)

A customer support chatbot is an AI-powered program designed to automate interactions, answer customer inquiries, provide information, and resolve common issues, often available 24/7.


In [5]:
import json, re

def parse_json(text):
    cleaned = re.sub(r"```(?:json)?", "", text).strip()
    return json.loads(cleaned)

In [7]:
POLICY = """
ShopWave Support Policy:
- Refunds: full refund within 30 days of delivery if the item is unused or defective.
- Damaged or defective items: free replacement or refund, customer does NOT need to return the item.
- Late deliveries: if more than 5 days late, customer gets a $10 store credit.
- Order changes: can be made within 2 hours of ordering.
- Escalate to a human if the customer is very angry, mentions legal action, or the issue is over $500.
"""

In [8]:
def step1_classify(ticket):
    prompt = f"""You are a support ticket classifier.
Read the customer message and return ONLY valid JSON with these keys:
- "intent": one of ["refund", "damaged_item", "late_delivery", "order_change", "product_question", "other"]
- "sentiment": one of ["positive", "neutral", "frustrated", "angry"]
- "urgency": one of ["low", "medium", "high"]

Customer message:
\"\"\"{ticket}\"\"\"
"""
    return parse_json(ask_llm(prompt, temperature=0))

# Test it
test_ticket = "Hi, I'm Maria. My order #48213 arrived yesterday but the blender jar is cracked. Can I get a new one?"
print(step1_classify(test_ticket))

{'intent': 'damaged_item', 'sentiment': 'frustrated', 'urgency': 'medium'}


In [10]:
def step2_extract(ticket, classification):
    prompt = f"""You extract facts from support tickets.
The ticket was classified as: {json.dumps(classification)}

Return ONLY valid JSON with these keys (use null if not mentioned):
- "customer_name"
- "order_number"
- "product"
- "problem_summary": one sentence
- "customer_wants": what outcome the customer is asking for

Customer message:
\"\"\"{ticket}\"\"\"
"""
    return parse_json(ask_llm(prompt, temperature=0))

In [16]:
def step3_draft(classification, details):
    prompt = f"""You are a friendly, concise ShopWave support agent.

Ticket classification: {json.dumps(classification)}
Ticket details: {json.dumps(details)}

Company policy:
{POLICY}

Write a reply to the customer that:
1. Greets them by name if known
2. Acknowledges their issue (match empathy to their sentiment)
3. Offers ONLY what the policy allows
4. Gives a clear next step
5. Stays under 120 words

If the policy says to escalate, say a specialist will follow up within 24 hours.
"""
    return ask_llm(prompt, temperature=0.5)

In [11]:
def step4_review(draft, classification, details):
    prompt = f"""You are a support QA reviewer. Check this draft reply.

Policy:
{POLICY}
Classification: {json.dumps(classification)}
Details: {json.dumps(details)}

Draft:
\"\"\"{draft}\"\"\"

Check: Does it follow policy? Promise anything not allowed? Is the tone right? Under 120 words?
Return ONLY valid JSON:
- "approved": true or false
- "issues": list of problems (empty list if none)
- "final_reply": the corrected reply (or the same draft if it was fine)
"""
    return parse_json(ask_llm(prompt, temperature=0))

In [12]:
def run_support_chain(ticket):
    print("=" * 70)
    print("CUSTOMER:", ticket)
    c = step1_classify(ticket)
    print("\n[Step 1] Classification:", c)
    d = step2_extract(ticket, c)
    print("[Step 2] Details:", d)
    draft = step3_draft(c, d)
    print("\n[Step 3] Draft:\n", draft)
    review = step4_review(draft, c, d)
    print("\n[Step 4] Approved:", review["approved"], "| Issues:", review["issues"])
    print("\nFINAL REPLY:\n", review["final_reply"])

In [19]:
tickets = [
    "Hi, I'm Maria. My order #48213 arrived yesterday but the blender jar is cracked. Can I get a new one?",
    "This is ridiculous. Order 99102 was supposed to arrive a WEEK ago and I still have nothing. Where is my headset??",
    "Hey, quick question: does the AeroFit water bottle keep drinks cold for 24 hours?",
]

for t in tickets:
    run_support_chain(t)

CUSTOMER: Hi, I'm Maria. My order #48213 arrived yesterday but the blender jar is cracked. Can I get a new one?

[Step 1] Classification: {'intent': 'damaged_item', 'sentiment': 'frustrated', 'urgency': 'medium'}


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 21.515039253s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},